# Data Science Challenge – Town A (Seasonal / H3N2 flu vaccine)
**Team:** UH-RMDS-DataChallenge-2026 · **Target:** `seasonal_vaccine`

**Goal:** decide who should receive a vaccine so that the town's **total cost** (vaccines + hospitalisations) is as low as possible.

Notebook sections: 0 Config · 1 Load · 2 Non-graphical EDA · 3 Graphical EDA · 4 Cost metric · 5 Pre-processing · 6 Models & cross-validation · 7 Tuning · 8 Results & ranking · 9 Submission files · 10 XAI

## 0. Configuration
Edit the paths below. Use the **Raw** GitHub URLs of the files uploaded to our repo.

In [ ]:
# ---- File locations (paste the GitHub 'Raw' URLs here) ----
# Upload the 3 CSVs to the repo with EXACTLY these names (no " (1)") so these links work for everyone
BASE = "https://raw.githubusercontent.com/UH-RMDS-DataChallenge-2026/data-science-challenge/main/"
TRAIN_PATH   = BASE + "dataset_A_training.csv"              # past survey (features + target)
LABELS_PATH  = None                                         # target is already inside the training file
TEST_PATH    = BASE + "dataset_A_testing.csv"               # people we must predict for (no target)
EXAMPLE_PATH = BASE + "dataset_A_example_submission.csv"    # required output format

GROUP_NUMBER = 0                 # <-- our group number, used in the submission file names
TARGET  = "seasonal_vaccine"     # target variable for Town A
ID_COL  = "respondent_id"        # unique identifier column

# ---- Costs for Town A (from the hackathon brief) ----
VACCINE_COST  = 300              # $ per vaccine (also paid in cash if someone wants one but doesn't get it)
HOSPITAL_COST = 18_500           # $ per hospitalisation for H3N2 flu

# ---- Assumptions for hospitalisation risk (researched with generative AI + CDC burden estimates) ----
VE = 0.40                        # vaccine effectiveness against hospitalisation (CDC: ~41-61%, H3N2 lower end)

RANDOM_STATE = 42                # set everywhere for reproducibility
N_FOLDS = 5                      # folds for cross-validation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, log_loss

sns.set_theme(style="whitegrid")     # consistent, readable plot style
np.random.seed(RANDOM_STATE)         # reproducible numpy randomness

## 1. Load the data

In [ ]:
train = pd.read_csv(TRAIN_PATH)                     # training survey
test  = pd.read_csv(TEST_PATH)                      # people to predict for
example = pd.read_csv(EXAMPLE_PATH)                 # required submission format

# If the target lives in a separate labels file, merge it in on the ID column
if TARGET not in train.columns and LABELS_PATH is not None:
    labels = pd.read_csv(LABELS_PATH)
    train = train.merge(labels[[ID_COL, TARGET]], on=ID_COL, how="left")

print("Train:", train.shape, "| Test:", test.shape)
print("Example submission columns:", list(example.columns))
example.head()

## 2. Non-graphical EDA

In [ ]:
# Column types and non-null counts
train.info()

In [ ]:
# Percentage of missing values per column (sorted) – decides our imputation strategy
missing_pct = train.isnull().mean().mul(100).sort_values(ascending=False)
missing_pct[missing_pct > 0].round(2)

In [ ]:
# Class balance of the target – are most people vaccine takers or refusers?
print(train[TARGET].value_counts(normalize=True).round(3))

In [ ]:
# Summary statistics of the numeric columns
train.describe().T

In [ ]:
# Distinct values of each categorical (text) column – spot odd labels and disguised missing values
for col in train.select_dtypes(include="object").columns:
    print(col, "->", train[col].unique()[:10])

## 3. Graphical EDA

In [ ]:
# Missing values per column as a bar chart
missing_pct[missing_pct > 0].plot(kind="barh", figsize=(7, 8), color="tab:orange")
plt.xlabel("% missing"); plt.title("Missing values per feature"); plt.tight_layout(); plt.show()

In [ ]:
# Vaccine uptake rate for the features we expect to matter most
key_feats = ["age_group", "doctor_recc_seasonal", "opinion_seas_risk", "opinion_seas_vacc_effective",
             "health_insurance", "number_chronic_med_conditions"]
key_feats = [f for f in key_feats if f in train.columns]      # keep only those present

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, f in zip(axes.ravel(), key_feats):
    rate = train.groupby(f)[TARGET].mean()                     # fraction vaccinated in each category
    rate.plot(kind="bar", ax=ax, color="tab:blue")
    ax.set_title(f"Vaccine uptake by {f}"); ax.set_ylabel("Uptake rate"); ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
# Correlation heatmap of numeric features with the target
num_cols_eda = train.select_dtypes(include="number").columns.drop(ID_COL, errors="ignore")
plt.figure(figsize=(12, 10))
sns.heatmap(train[num_cols_eda].corr(), cmap="coolwarm", center=0)
plt.title("Correlation between numeric features"); plt.show()

## 4. The cost metric (task-specific evaluation)
Everyone gets infected. For each person with hospitalisation probability *h* (if unvaccinated):

| | Wants vaccine (y=1) | Does not want it (y=0) |
|---|---|---|
| **Sent (pred=1)** | 300 + h·(1−VE)·18,500 | 300 + h·18,500 (wasted vaccine) |
| **Not sent (pred=0)** | 300 + h·18,500 (town pays cash, person unprotected) | h·18,500 |

**Total cost = sum over people. Lower is better.**

**Hospitalisation probability per infection** (CDC hospitalisations ÷ symptomatic illnesses, H3N2 season 2017-18): 18–49 ≈ 0.56 %, 50–64 ≈ 1.06 %, 65+ ≈ 9 % (to cross-check). Chronic conditions multiply risk by (1 + number of conditions) – a stated assumption.

**Decision rule:** send a vaccine if P(wants) > 300 / (300 + 18,500·h·VE). The default 0.5 threshold is not optimal here.

In [ ]:
def age_risk(age_label):
    """Baseline hospitalisation probability per infection for an age-group label."""
    s = str(age_label)
    if "65" in s and "64" not in s:     # '65+ Years'
        return 0.09
    if s.startswith("55"):              # '55 - 64 Years'
        return 0.0106
    if s.startswith("45"):              # '45 - 54 Years' (between the two CDC bands)
        return 0.008
    return 0.0056                       # '18 - 34' / '35 - 44' (and unknown -> lowest risk)

def hosp_prob(df):
    """Per-person hospitalisation probability if unvaccinated: age risk x (1 + chronic conditions)."""
    base = df["age_group"].map(age_risk) if "age_group" in df else pd.Series(0.0056, index=df.index)
    n_chronic = df["number_chronic_med_conditions"].fillna(0) if "number_chronic_med_conditions" in df else 0
    return (base * (1 + n_chronic)).clip(upper=1.0).to_numpy()

def total_cost(y_true, y_pred, h):
    """Town's total $ cost for a set of send(1)/don't-send(0) decisions."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    cost = np.where(y_pred == 1,
                    VACCINE_COST + np.where(y_true == 1, h * (1 - VE), h) * HOSPITAL_COST,   # sent
                    np.where(y_true == 1, VACCINE_COST, 0) + h * HOSPITAL_COST)            # not sent
    return cost.sum()

def personal_threshold(h):
    """Probability above which sending a vaccine is worth it for each person."""
    return VACCINE_COST / (VACCINE_COST + HOSPITAL_COST * h * VE)

h_train = hosp_prob(train)
print(f"Mean hospitalisation probability: {h_train.mean():.4f}")
print("Personal thresholds range:", personal_threshold(h_train).min().round(3), "-", personal_threshold(h_train).max().round(3))

In [ ]:
# Baselines: what does the town pay with no model at all? (cost per person, $)
y_all = train[TARGET].to_numpy()
n = len(y_all)
baselines = {
    "Send to nobody":     total_cost(y_all, np.zeros(n), h_train) / n,
    "Send to everybody":  total_cost(y_all, np.ones(n),  h_train) / n,
    "Perfect knowledge":  total_cost(y_all, y_all,       h_train) / n,   # lower bound
}
pd.Series(baselines).round(2).to_frame("cost per person ($)")

## 5. Pre-processing pipeline
All steps sit inside a `Pipeline`, so they are **fitted on training folds only** (no data leakage).

In [ ]:
X = train.drop(columns=[TARGET, ID_COL])         # features
y = train[TARGET].astype(int)                     # target
X_test = test.drop(columns=[ID_COL])

cat_cols = X.select_dtypes(include="object").columns.tolist()   # text categories -> one-hot
num_cols = X.select_dtypes(exclude="object").columns.tolist()   # numeric / ordinal -> impute (+ scale)
print(len(cat_cols), "categorical |", len(num_cols), "numeric")

# Opinion scales use 3 = "Don't know", which is not really between 2 and 4 -> option to one-hot encode them
OPINION_COLS = [c for c in ["opinion_seas_vacc_effective", "opinion_seas_risk", "opinion_seas_sick_from_vacc"] if c in num_cols]

def make_preprocessor(num_imputer="median", scale=True, onehot_opinions=False, missing_indicator=False):
    """Build the column transformer.
    num_imputer: 'median' or 'knn' (compare imputation methods)
    onehot_opinions: treat the opinion scales as categories instead of numbers
    missing_indicator: add 0/1 'was missing' columns (useful because health_insurance is 40% missing)"""
    num_used = [c for c in num_cols if not (onehot_opinions and c in OPINION_COLS)]
    imputer = (KNNImputer(n_neighbors=5, add_indicator=missing_indicator) if num_imputer == "knn"
               else SimpleImputer(strategy="median", add_indicator=missing_indicator))
    num_steps = [("impute", imputer)] + ([("scale", StandardScaler())] if scale else [])
    cat_pipe = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="Missing")),  # 'Missing' as its own category
                         ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    parts = [("num", Pipeline(num_steps), num_used), ("cat", cat_pipe, cat_cols)]
    if onehot_opinions:   # missing opinion -> its own category (-1), then one-hot
        op_pipe = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value=-1)),
                            ("onehot", OneHotEncoder(handle_unknown="ignore"))])
        parts.append(("opinions", op_pipe, OPINION_COLS))
    return ColumnTransformer(parts)

## 6. Models and cross-validation
Out-of-fold probabilities let us compute the **cost** for three decision strategies per model: fixed 0.5, best global threshold, and personal (risk-based) threshold.

In [ ]:
models = {
    "LogReg (median)": Pipeline([("prep", make_preprocessor("median")),
                                 ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))]),
    "LogReg (KNN impute)": Pipeline([("prep", make_preprocessor("knn")),
                                     ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))]),
    "LogReg (opinions one-hot + missing flags)": Pipeline([("prep", make_preprocessor("median", onehot_opinions=True, missing_indicator=True)),
                                     ("clf", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))]),
    "Random Forest": Pipeline([("prep", make_preprocessor("median", scale=False)),
                               ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                                              n_jobs=-1, random_state=RANDOM_STATE))]),
    "HistGradBoost": Pipeline([("prep", make_preprocessor("median", scale=False)),
                               ("clf", HistGradientBoostingClassifier(random_state=RANDOM_STATE))]),
}

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
GRID = np.linspace(0.05, 0.95, 91)                 # candidate global thresholds

def evaluate(name, model):
    """Cross-validate one model; return per-fold costs ($/person) for each decision strategy."""
    rows = []
    for fold, (tr, va) in enumerate(skf.split(X, y)):
        model.fit(X.iloc[tr], y.iloc[tr])                         # fit on the training fold only
        p_tr = model.predict_proba(X.iloc[tr])[:, 1]              # probabilities on the training fold
        p_va = model.predict_proba(X.iloc[va])[:, 1]              # probabilities on unseen fold
        h_tr, h_va = h_train[tr], h_train[va]
        # choose the global threshold on the TRAINING fold (avoids optimistic bias), apply it to validation
        best_t = GRID[np.argmin([total_cost(y.iloc[tr], (p_tr > t).astype(int), h_tr) for t in GRID])]
        nv = len(va)
        rows.append({
            "model": name, "fold": fold, "auc": roc_auc_score(y.iloc[va], p_va),
            "logloss": log_loss(y.iloc[va], p_va), "best_global_t": best_t,
            "cost_0.5":      total_cost(y.iloc[va], (p_va > 0.5).astype(int), h_va) / nv,
            "cost_global_t": total_cost(y.iloc[va], (p_va > best_t).astype(int), h_va) / nv,
            "cost_personal": total_cost(y.iloc[va], (p_va > personal_threshold(h_va)).astype(int), h_va) / nv,
        })
    return rows

cv_rows = []
for name, model in models.items():
    print("Cross-validating:", name)
    cv_rows += evaluate(name, model)
cv = pd.DataFrame(cv_rows)
cv.groupby("model").agg(["mean", "std"]).round(3)

## 7. Hyper-parameter tuning (gradient boosting)
Tuned on **log-loss**, because the cost decisions rely on well-calibrated probabilities.

In [ ]:
hgb = Pipeline([("prep", make_preprocessor("median", scale=False)),
                ("clf", HistGradientBoostingClassifier(random_state=RANDOM_STATE))])
param_dist = {
    "clf__learning_rate":   [0.02, 0.05, 0.1],
    "clf__max_depth":       [None, 3, 5, 8],
    "clf__max_leaf_nodes":  [15, 31, 63],
    "clf__min_samples_leaf":[20, 50, 100],
    "clf__l2_regularization":[0.0, 0.1, 1.0],
}
search = RandomizedSearchCV(hgb, param_dist, n_iter=20, scoring="neg_log_loss",
                            cv=skf, n_jobs=-1, random_state=RANDOM_STATE)
search.fit(X, y)
print("Best params:", search.best_params_)
print("Best CV log-loss:", -search.best_score_)

models["HistGradBoost (tuned)"] = search.best_estimator_
cv = pd.concat([cv, pd.DataFrame(evaluate("HistGradBoost (tuned)", search.best_estimator_))], ignore_index=True)

## 8. Results and ranking of candidate submissions

In [ ]:
# One candidate = (model, decision strategy). Rank all candidates by mean CV cost per person.
long = cv.melt(id_vars=["model", "fold"], value_vars=["cost_0.5", "cost_global_t", "cost_personal"],
               var_name="strategy", value_name="cost")
ranking = (long.groupby(["model", "strategy"])["cost"].agg(["mean", "std"])
               .sort_values("mean").reset_index())
ranking["saving_vs_nobody_$pp"] = baselines["Send to nobody"] - ranking["mean"]
ranking.round(2)

In [ ]:
# Plot: mean cost per person (+/- std over folds) for every candidate, with baselines for context
fig, ax = plt.subplots(figsize=(9, 7))
labels = ranking["model"] + " | " + ranking["strategy"]
ax.barh(labels, ranking["mean"], xerr=ranking["std"], color="tab:blue")
for name, val in baselines.items():
    ax.axvline(val, ls="--", label=name, color={"Send to nobody": "red", "Send to everybody": "orange",
                                               "Perfect knowledge": "green"}[name])
ax.invert_yaxis(); ax.set_xlabel("Cost per person ($) – lower is better"); ax.legend(); plt.tight_layout(); plt.show()

### Sensitivity check: does our choice depend on the assumptions?
The hospitalisation probabilities and VE are estimates. We recompute the cost under different assumptions; if the best strategy stays the same, our ranking is robust.

In [ ]:
from sklearn.model_selection import cross_val_predict

def cost_pp(y_true, y_pred, h, ve):
    """Cost per person for a given vaccine-effectiveness value (same logic as total_cost)."""
    return np.where(y_pred == 1, VACCINE_COST + np.where(y_true == 1, h * (1 - ve), h) * HOSPITAL_COST,
                    np.where(y_true == 1, VACCINE_COST, 0) + h * HOSPITAL_COST).mean()

# Out-of-fold probabilities for the top-3 models (each person predicted by a model that never saw them)
oof = {m: cross_val_predict(models[m], X, y, cv=skf, method="predict_proba")[:, 1]
       for m in ranking["model"].drop_duplicates().head(3)}

base_age = train["age_group"].map(age_risk)
chronic = train["number_chronic_med_conditions"].fillna(0)
scenarios = {   # name: (VE, risk for 65+, chronic multiplier per condition)
    "Base (VE 0.4, 65+ 9%, x(1+n))": (0.40, 0.09, 1.0),
    "Lower VE 0.3":                  (0.30, 0.09, 1.0),
    "Higher VE 0.5":                 (0.50, 0.09, 1.0),
    "65+ risk 5%":                   (0.40, 0.05, 1.0),
    "Weaker chronic effect x(1+0.5n)": (0.40, 0.09, 0.5),
}
rows = []
for sc, (ve, r65, cm) in scenarios.items():
    h = (base_age.where(~train["age_group"].astype(str).str.startswith("65"), r65) * (1 + cm * chronic)).clip(upper=1).to_numpy()
    t_pers = VACCINE_COST / (VACCINE_COST + HOSPITAL_COST * h * ve)          # personal thresholds under this scenario
    row = {"scenario": sc, "send to nobody": cost_pp(y, np.zeros(len(y)), h, ve),
           "send to everybody": cost_pp(y, np.ones(len(y)), h, ve)}
    for m, p in oof.items():
        row[f"{m} | personal"] = cost_pp(y, (p > t_pers).astype(int), h, ve)
        row[f"{m} | 0.5"] = cost_pp(y, (p > 0.5).astype(int), h, ve)
    rows.append(row)
pd.DataFrame(rows).set_index("scenario").round(1).T

## 9. Create the 5 submission files
Refit each of the top-5 candidates on **all** training data, then predict the test set.

In [ ]:
top5 = ranking.head(5)
pred_col = [c for c in example.columns if c != ID_COL][0]     # name of the prediction column in the example file
h_test = hosp_prob(test)

for order, row in enumerate(top5.itertuples(), start=1):
    model = models[row.model]
    model.fit(X, y)                                             # refit on the full training set
    p_test = model.predict_proba(X_test)[:, 1]
    if row.strategy == "cost_0.5":
        t = 0.5
    elif row.strategy == "cost_global_t":
        t = cv.loc[cv.model == row.model, "best_global_t"].median()   # typical threshold chosen in CV
    else:
        t = personal_threshold(h_test)                          # risk-based threshold per person
    sub = pd.DataFrame({ID_COL: test[ID_COL], pred_col: (p_test > t).astype(int)})
    fname = f"challenge_submission_group_{GROUP_NUMBER}_order_{order}.csv"
    sub.to_csv(fname, index=False)
    print(f"{fname}: {row.model} | {row.strategy} | CV cost ${row.mean:.2f}/person | send rate {sub[pred_col].mean():.2%}")

In [ ]:
# Download the files from Colab (also upload them to GitHub)
from google.colab import files
for order in range(1, 6):
    files.download(f"challenge_submission_group_{GROUP_NUMBER}_order_{order}.csv")

## 10. Explainable AI (can be continued after the hackathon)

In [ ]:
from sklearn.inspection import permutation_importance

best_tree = models["HistGradBoost (tuned)"]
best_tree.fit(X, y)
# Permutation importance: drop in AUC when each feature is shuffled (sample for speed)
sample = X.sample(min(3000, len(X)), random_state=RANDOM_STATE)
pi = permutation_importance(best_tree, sample, y.loc[sample.index], scoring="roc_auc",
                            n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=sample.columns).sort_values()
imp.tail(15).plot(kind="barh", figsize=(8, 6), title="Permutation importance (AUC drop)"); plt.show()

In [ ]:
# SHAP values for the tuned gradient boosting model (global + one individual explanation)
import shap
prep, clf = best_tree.named_steps["prep"], best_tree.named_steps["clf"]
X_enc = prep.transform(sample)
X_enc = X_enc.toarray() if hasattr(X_enc, "toarray") else X_enc     # dense array for SHAP
feat_names = prep.get_feature_names_out()
explainer = shap.TreeExplainer(clf)
shap_values = explainer(pd.DataFrame(X_enc, columns=feat_names))
shap.plots.beeswarm(shap_values, max_display=15)     # global importance + direction
shap.plots.waterfall(shap_values[0])                  # why the model decided for one person